# 01 构造超图（含带权超边）

本 notebook 演示 `honet.Hypergraph` / `honet.Hyperedge` 的基本构造方式：

- 结点可以是任意可哈希对象（这里用研究者姓名）；
- 一条**超边**表示一次多元交互（一篇论文由 2～4 位作者共同完成）；
- 超边可以带 `label`（论文编号）和 `weight`（例如合作强度／被引次数）。

**预期看到**：4 个结点、3 条超边；其中一条权重为 3.0 的超边，以及按超边条数统计的高阶度。

运行方式：`uv run --with jupyter --with ipykernel jupyter lab examples/`

In [1]:
import honet

# 硬编码的小规模协作数据：结点=研究者，超边=一篇论文
hg = honet.Hypergraph(
    nodes=["alice", "bob", "carol", "dave"],
    edges=[
        ["alice", "bob", "carol"],  # 三元合作
        ["bob", "carol"],  # 二元合作
    ],
)

# 带权超边：同一批作者反复合作，用 weight 表达强度
hg.add_edge(["alice", "carol", "dave"], label="paper-2024", weight=3.0)

print("结点顺序:", hg.nodes)
print("规模:", hg.num_nodes, "个结点,", hg.num_edges, "条超边")
print("是否均匀超图:", hg.is_uniform, "| 各边大小:", hg.edge_sizes)

结点顺序: ('alice', 'bob', 'carol', 'dave')
规模: 4 个结点, 3 条超边
是否均匀超图: False | 各边大小: (3, 2, 3)


## 查看每条超边的字段

`Hyperedge` 是不可变对象，包含三个字段：`members`（成员集合）、`label`（可选标签）、`weight`（权重，默认 1.0）。
把 `members` 排序后打印，只是为了输出稳定、便于阅读；内部始终是无序集合。

In [2]:
for i, edge in enumerate(hg.edges):
    members = ", ".join(sorted(edge.members))
    print(
        f"边{i}: members={{{members}}}  size={edge.size}  "
        f"label={edge.label!r}  weight={edge.weight}"
    )

print()
print("alice 的高阶度（参与的超边条数）:", hg.degree("alice"))
print("alice 的共边邻居:", sorted(hg.neighbors("alice")))
print("dave 的共边邻居:", sorted(hg.neighbors("dave")))

边0: members={alice, bob, carol}  size=3  label=None  weight=1.0
边1: members={bob, carol}  size=2  label=None  weight=1.0
边2: members={alice, carol, dave}  size=3  label='paper-2024'  weight=3.0

alice 的高阶度（参与的超边条数）: 2
alice 的共边邻居: ['bob', 'carol', 'dave']
dave 的共边邻居: ['alice', 'carol']


## 结构性约束

成员集合与标签完全相同的超边会被拒绝（避免静默覆盖数据）；权重非法（≤0）也会在构造时立刻报错。
下面用 `try/except` 演示这两类保护。

In [3]:
try:
    hg.add_edge(["carol", "bob"])  # 与边1成员相同（顺序无关）、标签同为 None
except ValueError as exc:
    print("重复超边被拒绝 ->", exc)

try:
    hg.add_edge(["alice", "dave"], weight=0)
except ValueError as exc:
    print("非法权重被拒绝 ->", exc)

print()
print("最终图:", hg)

重复超边被拒绝 -> 重复的超边：['bob', 'carol']（label=None）
非法权重被拒绝 -> 超边权重必须为正数：0

最终图: <Hypergraph nodes=4 edges=3 uniform=False>
